In [1]:
"""
Cluster Characterization Analysis
Analyzes the climate characteristics of each discovered cluster
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

print("="*80)
print("CLUSTER CHARACTERIZATION ANALYSIS")
print("="*80)

# Load results
print("\nLoading data...")
df_results = pd.read_csv('preprocessed_data/clustering_results_495locs.csv')
df_features = pd.read_csv('preprocessed_data/engineered_features_495locs.csv')

print(f"✓ Loaded {len(df_results)} locations")
print(f"✓ Loaded {len(df_features.columns)} features")

# Get available columns
available_cols = df_features.columns.tolist()
print(f"\nFirst 20 available columns:")
for i, col in enumerate(available_cols[:20], 1):
    print(f"  {i}. {col}")

# Define key parameters to analyze (flexible - use what's available)
key_params = {}

# Temperature parameters
for param in ['T2M_mean', 'T2M_std', 'T2M_MIN_mean', 'T2M_MAX_mean']:
    if param in available_cols:
        key_params['Temperature'] = param
        break

# Precipitation parameters
for param in ['PRECTOTCORR_mean', 'PRECTOT_mean']:
    if param in available_cols:
        key_params['Precipitation'] = param
        break

# Humidity parameters
for param in ['RH2M_mean', 'RH_mean']:
    if param in available_cols:
        key_params['Humidity'] = param
        break

# Solar radiation parameters
for param in ['ALLSKY_SFC_SW_DWN_std', 'ALLSKY_SFC_SW_DNI_mean', 
              'ALLSKY_KT_mean', 'MIDDAY_INSOL_mean']:
    if param in available_cols:
        key_params['Solar Radiation'] = param
        break

# Wind parameters
for param in ['WS2M_mean', 'WS10M_mean', 'WS_mean']:
    if param in available_cols:
        key_params['Wind Speed'] = param
        break

# Cloud parameters
for param in ['CLOUD_AMT_mean', 'CLOUD_OD_mean']:
    if param in available_cols:
        key_params['Cloud Cover'] = param
        break

print(f"\n✓ Using these key parameters:")
for name, col in key_params.items():
    print(f"  {name}: {col}")

# =============================================================================
# CLUSTER ANALYSIS
# =============================================================================

n_clusters = df_features['cluster'].nunique()
print(f"\n" + "="*80)
print(f"ANALYZING {n_clusters} CLUSTERS")
print("="*80)

cluster_summaries = []

for cluster_id in sorted(df_features['cluster'].unique()):
    cluster_data = df_features[df_features['cluster'] == cluster_id]
    
    print(f"\n{'='*80}")
    print(f"CLUSTER {cluster_id}: {len(cluster_data)} locations ({len(cluster_data)/len(df_features)*100:.1f}%)")
    print(f"{'='*80}")
    
    # Geographic extent
    lat_min, lat_max = cluster_data['lat'].min(), cluster_data['lat'].max()
    lon_min, lon_max = cluster_data['lon'].min(), cluster_data['lon'].max()
    lat_center = cluster_data['lat'].mean()
    lon_center = cluster_data['lon'].mean()
    
    print(f"\n📍 Geographic Location:")
    print(f"  Latitude range:  {lat_min:.2f}° to {lat_max:.2f}°N (center: {lat_center:.2f}°)")
    print(f"  Longitude range: {lon_min:.2f}° to {lon_max:.2f}°W (center: {lon_center:.2f}°)")
    
    # Determine region based on location
    if lon_center > -121:
        region = "Eastern Washington"
    elif lon_center > -122.5:
        region = "Central/Cascade Region"
    else:
        region = "Western Washington"
    print(f"  Primary region: {region}")
    
    # Climate characteristics
    print(f"\n🌡️  Climate Characteristics:")
    summary = {'cluster': cluster_id, 'n_locations': len(cluster_data), 
               'lat_center': lat_center, 'lon_center': lon_center, 'region': region}
    
    for param_name, param_col in key_params.items():
        value = cluster_data[param_col].mean()
        summary[param_name] = value
        
        if param_name == 'Temperature':
            print(f"  {param_name}: {value:.1f}°C")
        elif param_name == 'Precipitation':
            print(f"  {param_name}: {value:.2f} mm/day")
        elif param_name == 'Humidity':
            print(f"  {param_name}: {value:.1f}%")
        elif param_name == 'Solar Radiation':
            if 'KT' in param_col:
                print(f"  {param_name} (Clearness): {value:.3f}")
            else:
                print(f"  {param_name}: {value:.1f}")
        elif param_name == 'Wind Speed':
            print(f"  {param_name}: {value:.2f} m/s")
        elif param_name == 'Cloud Cover':
            print(f"  {param_name}: {value:.1f}%")
    
    cluster_summaries.append(summary)

# =============================================================================
# CREATE SUMMARY DATAFRAME
# =============================================================================

df_summary = pd.DataFrame(cluster_summaries)
print(f"\n" + "="*80)
print("CLUSTER SUMMARY TABLE")
print("="*80)
print(df_summary.to_string(index=False))

# Save summary
output_file = "preprocessed_data/cluster_summary.csv"
df_summary.to_csv(output_file, index=False)
print(f"\n✓ Saved cluster summary to: {output_file}")

# =============================================================================
# VISUALIZATIONS
# =============================================================================

print(f"\n" + "="*80)
print("CREATING VISUALIZATIONS")
print("="*80)

os.makedirs("preprocessed_data/visualizations", exist_ok=True)

# 1. Cluster comparison bar chart for numeric parameters
numeric_params = [k for k in key_params.keys() if k in df_summary.columns]

if len(numeric_params) >= 3:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    for idx, param in enumerate(numeric_params[:6]):
        if idx < len(axes):
            ax = axes[idx]
            data = df_summary.sort_values('cluster')
            ax.bar(data['cluster'], data[param], color='steelblue', alpha=0.7, edgecolor='black')
            ax.set_xlabel('Cluster ID', fontsize=11)
            ax.set_ylabel(param, fontsize=11)
            ax.set_title(f'{param} by Cluster', fontsize=12, fontweight='bold')
            ax.grid(axis='y', alpha=0.3)
            ax.set_xticks(data['cluster'])
    
    # Hide unused subplots
    for idx in range(len(numeric_params), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig('preprocessed_data/visualizations/cluster_comparison.png', dpi=150, bbox_inches='tight')
    print("✓ Saved: cluster_comparison.png")
    plt.close()

# 2. Radar chart for cluster profiles (select 6 representative clusters)
if len(numeric_params) >= 4:
    # Select up to 6 clusters for visualization
    selected_clusters = sorted(df_summary['cluster'].unique())[:6]
    
    fig = plt.figure(figsize=(15, 10))
    
    for plot_idx, cluster_id in enumerate(selected_clusters, 1):
        ax = fig.add_subplot(2, 3, plot_idx, projection='polar')
        
        cluster_row = df_summary[df_summary['cluster'] == cluster_id].iloc[0]
        
        # Get values and normalize to 0-1 range for radar chart
        values = []
        labels = []
        for param in numeric_params[:6]:  # Max 6 parameters for readability
            if param in cluster_row:
                val = cluster_row[param]
                # Normalize based on global min/max
                val_min = df_summary[param].min()
                val_max = df_summary[param].max()
                if val_max > val_min:
                    normalized = (val - val_min) / (val_max - val_min)
                else:
                    normalized = 0.5
                values.append(normalized)
                labels.append(param)
        
        # Close the plot
        values += values[:1]
        
        # Angles for each axis
        angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
        angles += angles[:1]
        
        ax.plot(angles, values, 'o-', linewidth=2, label=f'Cluster {cluster_id}')
        ax.fill(angles, values, alpha=0.25)
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(labels, size=8)
        ax.set_ylim(0, 1)
        ax.set_title(f'Cluster {cluster_id}\n({cluster_row["region"]})', 
                    fontsize=11, fontweight='bold', pad=20)
        ax.grid(True)
    
    plt.tight_layout()
    plt.savefig('preprocessed_data/visualizations/cluster_radar_profiles.png', dpi=150, bbox_inches='tight')
    print("✓ Saved: cluster_radar_profiles.png")
    plt.close()

# 3. Enhanced geographic map with labels
fig, ax = plt.subplots(figsize=(16, 12))

# Plot clusters
for cluster_id in sorted(df_features['cluster'].unique()):
    cluster_data = df_features[df_features['cluster'] == cluster_id]
    ax.scatter(cluster_data['lon'], cluster_data['lat'], 
              label=f'Cluster {cluster_id} (n={len(cluster_data)})',
              s=120, alpha=0.7, edgecolors='black', linewidth=0.5)

# Add cluster centroids with labels
for _, row in df_summary.iterrows():
    ax.annotate(f"{int(row['cluster'])}", 
               xy=(row['lon_center'], row['lat_center']),
               fontsize=10, fontweight='bold',
               ha='center', va='center',
               bbox=dict(boxstyle='circle', facecolor='white', edgecolor='black', alpha=0.8))

ax.set_xlabel('Longitude', fontsize=14)
ax.set_ylabel('Latitude', fontsize=14)
ax.set_title(f'Washington State Climate Microclimates (K={n_clusters})\nWith Cluster Centroids', 
            fontsize=16, fontweight='bold', pad=20)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('preprocessed_data/visualizations/cluster_map_detailed.png', dpi=150, bbox_inches='tight')
print("✓ Saved: cluster_map_detailed.png")
plt.close()

# 4. Cluster size distribution
fig, ax = plt.subplots(figsize=(12, 6))
cluster_sizes = df_summary.sort_values('cluster')
bars = ax.bar(cluster_sizes['cluster'], cluster_sizes['n_locations'], 
              color='steelblue', alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_xlabel('Cluster ID', fontsize=14)
ax.set_ylabel('Number of Locations', fontsize=14)
ax.set_title('Cluster Size Distribution', fontsize=16, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.set_xticks(cluster_sizes['cluster'])

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{int(height)}',
           ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('preprocessed_data/visualizations/cluster_sizes.png', dpi=150, bbox_inches='tight')
print("✓ Saved: cluster_sizes.png")
plt.close()

# =============================================================================
# SUMMARY
# =============================================================================

print(f"\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)

print(f"\n📊 Summary Statistics:")
print(f"  Total clusters: {n_clusters}")
print(f"  Total locations: {len(df_features)}")
print(f"  Average cluster size: {len(df_features) / n_clusters:.1f} locations")
print(f"  Largest cluster: {df_summary['n_locations'].max()} locations")
print(f"  Smallest cluster: {df_summary['n_locations'].min()} locations")

print(f"\n📁 Output Files:")
print(f"  • preprocessed_data/cluster_summary.csv")
print(f"  • preprocessed_data/visualizations/cluster_comparison.png")
print(f"  • preprocessed_data/visualizations/cluster_radar_profiles.png")
print(f"  • preprocessed_data/visualizations/cluster_map_detailed.png")
print(f"  • preprocessed_data/visualizations/cluster_sizes.png")

print(f"\n🌍 Regional Distribution:")
region_counts = df_summary['region'].value_counts()
for region, count in region_counts.items():
    print(f"  {region}: {count} clusters")

print("\n" + "="*80)

CLUSTER CHARACTERIZATION ANALYSIS

Loading data...
✓ Loaded 495 locations
✓ Loaded 187 features

First 20 available columns:
  1. lat
  2. lon
  3. T2M_mean
  4. T2M_std
  5. T2M_min
  6. T2M_max
  7. T2MDEW_mean
  8. T2MDEW_std
  9. T2MDEW_min
  10. T2MDEW_max
  11. T2MWET_mean
  12. T2MWET_std
  13. T2MWET_max
  14. T2M_MAX_mean
  15. T2M_MAX_std
  16. T2M_MAX_min
  17. T2M_MAX_max
  18. T2M_MIN_mean
  19. T2M_MIN_min
  20. T2M_MIN_max

✓ Using these key parameters:
  Temperature: T2M_mean
  Precipitation: PRECTOTCORR_mean
  Humidity: RH2M_mean
  Solar Radiation: ALLSKY_SFC_SW_DNI_mean
  Wind Speed: WS2M_mean
  Cloud Cover: CLOUD_AMT_mean

ANALYZING 14 CLUSTERS

CLUSTER 0: 42 locations (8.5%)

📍 Geographic Location:
  Latitude range:  47.78° to 49.03°N (center: 48.47°)
  Longitude range: -123.77° to -122.02°W (center: -122.93°)
  Primary region: Western Washington

🌡️  Climate Characteristics:
  Temperature: 9.6°C
  Precipitation: 3.71 mm/day
  Humidity: 83.7%
  Solar Radiation: 167.